In [0]:
from pyspark.sql.types import IntegerType, DoubleType, StringType, BooleanType
from pyspark.sql.functions import col, trim, regexp_replace

####Establish connection to the Datalake storage containers

In [0]:
spark.conf.set("fs.azure.account.auth.type.stinstacartlake.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.stinstacartlake.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.stinstacartlake.dfs.core.windows.net", "05b534b6-4a04-4d69-aa90-d1e381ef65f7")
spark.conf.set("fs.azure.account.oauth2.client.secret.stinstacartlake.dfs.core.windows.net", dbutils.secrets.get(scope="instacart-akv-scope", key="client-secret"))
spark.conf.set("fs.azure.account.oauth2.client.endpoint.stinstacartlake.dfs.core.windows.net", "https://login.microsoftonline.com/38c53c6c-2815-4635-9ecd-b14b896cf9a6/oauth2/token")

# Test the connection
# display(dbutils.fs.ls("abfss://bronze@stinstacartlake.dfs.core.windows.net/"))

####Apply Type Casting to DataFrames : aisles.csv

In [0]:
#Read data from Bronze layer
df_brz_aisles = spark.read.format("parquet") \
    .option("header", True) \
    .load("abfss://bronze@stinstacartlake.dfs.core.windows.net/aisles.parquet")

#Transform and cast columns for the Silver Layer
df_slv_aisles = df_brz_aisles \
    .withColumn("aisle_id", col("aisle_id").cast(IntegerType())) \
    .withColumn("aisle", col("aisle_id").cast(StringType()))

#Verify the updated schema
df_slv_aisles.printSchema()

root
 |-- aisle_id: integer (nullable = true)
 |-- aisle: string (nullable = true)



####Apply Type Casting to DataFrames : departments.csv

In [0]:
#Read data from Bronze layer
df_brz_dept = spark.read.format("parquet") \
    .option("header", True) \
    .load("abfss://bronze@stinstacartlake.dfs.core.windows.net/departments.parquet")

#Transform and cast columns for the Silver Layer
df_slv_dept = df_brz_dept \
    .withColumn("department_id", col("department_id").cast(IntegerType())) \
    .withColumn("department", col("department").cast(StringType()))

#Verify the updated schema
df_slv_dept.printSchema()

root
 |-- department_id: integer (nullable = true)
 |-- department: string (nullable = true)



####Apply Type Casting to DataFrames : order_products__prior.csv

In [0]:
#Read data from Bronze layer
df_brz_opp = spark.read.format("parquet") \
    .option("header", True) \
    .load("abfss://bronze@stinstacartlake.dfs.core.windows.net/order_products__prior.parquet")

#Transform and cast columns for the Silver Layer
df_slv_opp = df_brz_opp \
    .withColumn("order_id", col("order_id").cast(IntegerType())) \
    .withColumn("product_id", col("product_id").cast(IntegerType())) \
    .withColumn("add_to_cart_order", col("add_to_cart_order").cast(IntegerType())) \
    .withColumn("reordered", col("reordered").cast(BooleanType()))

#Verify the updated schema
df_slv_opp.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: boolean (nullable = true)



####Apply Type Casting to DataFrames : orders.csv

In [0]:
#Read data from Bronze layer
df_brz_ord = spark.read.format("parquet") \
    .option("header", True) \
    .load("abfss://bronze@stinstacartlake.dfs.core.windows.net/orders.parquet")

#Transform and cast columns for the Silver Layer
df_slv_ord = df_brz_ord \
    .withColumn("order_id", col("order_id").cast(IntegerType())) \
    .withColumn("user_id", col("user_id").cast(IntegerType())) \
    .withColumn("eval_set", col("eval_set").cast(StringType())) \
    .withColumn("order_number", col("order_number").cast(IntegerType())) \
    .withColumn("order_dow", col("order_dow").cast(IntegerType())) \
    .withColumn("order_hour_of_day", col("order_hour_of_day").cast(IntegerType())) \
    .withColumn("days_since_prior_order", col("days_since_prior_order").cast(DoubleType()).cast(IntegerType()))

#Verify the updated schema
df_slv_ord.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- eval_set: string (nullable = true)
 |-- order_number: integer (nullable = true)
 |-- order_dow: integer (nullable = true)
 |-- order_hour_of_day: integer (nullable = true)
 |-- days_since_prior_order: integer (nullable = true)



####Apply Type Casting to DataFrames : products.csv

In [0]:
#Read data from Bronze layer
df_brz_prd = spark.read.format("parquet") \
    .option("header", True) \
    .load("abfss://bronze@stinstacartlake.dfs.core.windows.net/products.parquet")

#Transform and cast columns for the Silver Layer
df_slv_prd = df_brz_prd \
    .withColumn("product_id", col("product_id").cast(IntegerType())) \
    .withColumn("product_name", col("product_name").cast(StringType())) \
    .withColumn("aisle_id", col("aisle_id").cast(IntegerType())) \
    .withColumn("department_id", col("department_id").cast(IntegerType()))

#Verify the updated schema
df_slv_prd.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- aisle_id: integer (nullable = true)
 |-- department_id: integer (nullable = true)



####Remove Duplicate Rows, if any

In [0]:

#Remove if any of the whole rows are repeated
df_slv_aisles = df_slv_aisles.dropDuplicates()
df_slv_dept = df_slv_dept.dropDuplicates()

#Remove row if any duplicate row for the composite key order_id, product_id
df_slv_opp = df_slv_opp.dropDuplicates(["order_id", "product_id"])

#Remove row if any duplicate row for the primary key order_id
df_slv_ord = df_slv_ord.dropDuplicates(["order_id"])

#Remove row if any duplicate row for the primary key product_id
df_slv_prd = df_slv_prd.dropDuplicates(["product_id"])

####Cleanup data

In [0]:
#Remove Leading and Trailing spaces and replace multiple spaces with a single space in products file name
df_slv_prd = df_slv_prd.withColumn(
    "product_name",
    trim(regexp_replace(col("product_name"), "\\s+", " "))
)

####Write the final silver dataframes to Silver layer container as Delta tables

In [0]:
# Write aisles to the Silver layer container as a Delta table
df_slv_aisles.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://silver@stinstacartlake.dfs.core.windows.net/aisles")

# Write departments to the Silver layer container as a Delta table
df_slv_dept.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://silver@stinstacartlake.dfs.core.windows.net/departments")

# Write order_products__prior to the Silver layer container as a Delta table
df_slv_opp.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://silver@stinstacartlake.dfs.core.windows.net/order_products__prior")

# Write orders to the Silver layer container as a Delta table
df_slv_ord.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://silver@stinstacartlake.dfs.core.windows.net/orders")

# Write products to the Silver layer container as a Delta table
df_slv_prd.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://silver@stinstacartlake.dfs.core.windows.net/products")

####